In [1]:
import os
from dotenv import load_dotenv
from langchain.chat_models import ChatOpenAI
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain
from langchain.agents import Tool, initialize_agent, AgentType
from langchain.tools import TavilySearchResults

# iT WILL USE THE memory of conversation
from langchain.memory import ConversationBufferMemory

# Load API's

load_dotenv(".env")
openai_api_key = os.getenv("OPENAI_API_KEY")
tavily_api_key = os.getenv("TAVILY_API_KEY")

#Initialise LLM



llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

#Tools

# Simple QA tool (LLMChain)
qa_prompt = PromptTemplate.from_template("Answer clearly: {question}")
qa_chain = LLMChain(llm=llm, prompt=qa_prompt)
qa_tool = Tool(
    name="Simple_QA",
    func=qa_chain.run,
    description="Answer the question without hallucination"
)

# Web search tool (Tavily)
search_tool = Tool(
    name="Web_Search",
    func=TavilySearchResults(max_results=3).run,
    description="Search the internet for current info"
)

tools = [qa_tool, search_tool]

#MEMORY UTILISATION
memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)

C:\Users\my pc\AppData\Local\Temp\ipykernel_10856\3575917623.py:22: LangChainDeprecationWarning: The class `ChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import ChatOpenAI``.
  llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)
C:\Users\my pc\AppData\Local\Temp\ipykernel_10856\3575917623.py:28: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use :meth:`~RunnableSequence, e.g., `prompt | llm`` instead.
  qa_chain = LLMChain(llm=llm, prompt=qa_prompt)
C:\Users\my pc\AppData\Local\Temp\ipykernel_10856\3575917623.py:38: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the

In [3]:
#1) Zero-Shot ReAct (simple + reliable default)

agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True
)
result = agent.run("Summarize LangChain in 2 lines, then tell me who created it.")



> Entering new AgentExecutor chain...
I should search the internet for information on LangChain to summarize it and find out who created it.
Action: Web_Search
Action Input: "LangChain summary and creator"
Observation: [{'title': 'LangChain overview - Docs by LangChain', 'url': 'https://docs.langchain.com/oss/python/langchain/overview', 'content': 'Create an agent\n Core benefits\n\n# LangChain overview\n\nLangChain is an open source framework with a pre-built agent architecture and integrations for any model or tool — so you can build agents that adapt as fast as the ecosystem evolves\n\nLangChain is the easy way to start building completely custom agents and applications powered by LLMs. With under 10 lines of code, you can connect to OpenAI, Anthropic, Google, and more. LangChain provides a pre-built agent architecture and model integrations to help you get started quickly and seamlessly incorporate LLMs into your agents and applications. [...] LangChain agents are built on top of

In [ ]:
result

#ERROR IN GPT 5 MODELS

BadRequestError: Error code: 400 - {'error': {'message': "Unsupported parameter: 'stop' is not supported with this model.", 'type': 'invalid_request_error', 'param': 'stop', 'code': 'unsupported_parameter'}}


In [2]:
#2) Conversational ReAct (with memory; non-chat model baseline)
agent = initialize_agent(
    tools=tools,
    llm=llm,
    #Memory support
    memory=memory,
    agent=AgentType.CONVERSATIONAL_REACT_DESCRIPTION,
    verbose=True
)
result = agent.run("Summarize LangChain in 2 lines, then tell me who created it.")


C:\Users\my pc\AppData\Local\Temp\ipykernel_10856\2972630844.py:2: LangChainDeprecationWarning: LangChain agents will continue to be supported, but it is recommended for new use cases to be built with LangGraph. LangGraph offers a more flexible and full-featured framework for building agents, including support for tool-calling, persistence of state, and human-in-the-loop workflows. For details, refer to the `LangGraph documentation <https://langchain-ai.github.io/langgraph/>`_ as well as guides for `Migrating from AgentExecutor <https://python.langchain.com/docs/how_to/migrate_agent/>`_ and LangGraph's `Pre-built ReAct agent <https://langchain-ai.github.io/langgraph/how-tos/create-react-agent/>`_.
  agent = initialize_agent(
C:\Users\my pc\AppData\Local\Temp\ipykernel_10856\2972630844.py:10: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  result = agent.run("Summarize LangChain in 2 lines, 



> Entering new AgentExecutor chain...
Thought: Do I need to use a tool? Yes
Action: Simple_QA
Action Input: Summarize LangChain in 2 lines, then tell me who created it.
Observation: LangChain is a blockchain platform that aims to connect language service providers with clients in a decentralized manner. It was created by a team of developers led by CEO and founder, Alexey Kuznetsov.
Thought:Do I need to use a tool? No
AI: LangChain is a blockchain platform connecting language service providers with clients in a decentralized way. It was created by a team led by CEO and founder, Alexey Kuznetsov.

> Finished chain.


In [ ]:
# The conversational React agent didnt use the tool when GPT 4o MINI WAS USED
# BUT IT USED THE TOOL WITH GPT 3.5

In [4]:
#3) Chat Conversational ReAct (recommended for GPT-3.5/4 chat)
agent = initialize_agent(
    tools=tools,
    llm=llm,
    memory=memory,
    agent=AgentType.CHAT_CONVERSATIONAL_REACT_DESCRIPTION,
    verbose=True
)
result = agent.run("Plan a 3-step study path for LangChain.")
#result = agent.run("Summarize LangChain in 2 lines, then tell me who created it.")




> Entering new AgentExecutor chain...
```json
{
    "action": "Web_Search",
    "action_input": "Plan a 3-step study path for LangChain"
}
```
Observation: [{'title': 'Teaching LangChain Agents to Plan & Run Multi-Step, Multi-Tool ...', 'url': 'https://medium.com/@avigoldfinger/teaching-langchain-agents-to-plan-run-multi-step-multi-tool-workflows-82ac908fd56e', 'content': '1. Define a single, verifiable business outcome. Resist “general assistant”; pick “reduce contract review time” or “surface missed lab results”.\n2. Wrap every external system as a LangChain tool. Provide a terse description and strict input schema.\n3. Draft a planner prompt that outputs a numbered plan referencing only the tools available.\n4. Add executor agents with the minimum tool set required.\n5. Insert an evaluator agent that can halt, reroute, or escalate.\n6. Trace everything in LangSmith — every thought, action, observation.\n7. Harvest good traces into a KnowAgent-style action library for few-shot plan

In [5]:
agent = initialize_agent(
    tools=tools,
    llm=ChatOpenAI(model="gpt-4o-mini", temperature=0),
    memory=memory,
    agent=AgentType.CHAT_CONVERSATIONAL_REACT_DESCRIPTION,
    verbose=True
)
result = agent.run("Plan a 3-step study path for LangChain.")
#result = agent.run("Summarize LangChain in 2 lines, then tell me who created it.")



> Entering new AgentExecutor chain...
```json
{
    "action": "Final Answer",
    "action_input": "1. **Understand the Basics**: Start by reading the official LangChain documentation to grasp its core concepts, architecture, and use cases. 2. **Hands-On Practice**: Build simple projects using LangChain to familiarize yourself with its tools and functionalities. 3. **Advanced Topics**: Explore advanced features, integrations, and best practices through tutorials, community forums, and case studies."
}
```

> Finished chain.


In [6]:
# 4. Structured chat Zero shot React decsription
# When tool requires multiple inputs
from langchain.tools import StructuredTool
from pydantic import BaseModel

# ✅ Define the input schema using Pydantic
class TitleInput(BaseModel):
    topic: str
    tone: str = "concise"

# ✅ Define the tool function
def title_tool_fn(topic: str, tone: str = "concise") -> str:
    return f"{tone.title()} Title: {topic} in Practice"

# ✅ Create the structured tool
title_tool = StructuredTool.from_function(
    name="TitleMaker",
    func=title_tool_fn,
    description="Generate a title given a topic and an optional tone.",
    args_schema=TitleInput
)

# ✅ Initialize the agent
agent = initialize_agent(
    tools=[title_tool],
    llm=llm,
    agent=AgentType.STRUCTURED_CHAT_ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True
)

# ✅ Run the agent
result = agent.run("Make a friendly title about LangGraph tutorials.")
print("✅ Result:", result)



> Entering new AgentExecutor chain...
Thought: I can use the TitleMaker tool to generate a friendly title about LangGraph tutorials.
Action:
```
{
  "action": "TitleMaker",
  "action_input": {"topic": "LangGraph tutorials", "tone": "friendly"}
}
```
Observation: Friendly Title: LangGraph tutorials in Practice
Thought:I know what to respond
Action:
```
{
  "action": "Final Answer",
  "action_input": "LangGraph tutorials in Practice"
}
```

> Finished chain.
✅ Result: LangGraph tutorials in Practice


The Workflow in Action:
User says: "Make a professional title about AI."

LLM thinks: "I should use TitleMaker. The user wants 'AI' as the topic and 'professional' as the tone."

LLM outputs JSON: {"topic": "AI", "tone": "professional"}.

LangChain validates: It checks the JSON against TitleInput. Everything looks good.

Logic runs: title_tool_fn("AI", "professional") is called.

Observation: The function returns "Professional Title: AI in Practice".

Final Response: The LLM sees that result and tells the user: "Here is your professional title: AI in Practice."

In [7]:
#5)OpenAI Functions (function-calling schema)
agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent=AgentType.OPENAI_FUNCTIONS,
    verbose=True
)
result = agent.run("Search the web for LangGraph docs and give me 3 bullets.")



> Entering new AgentExecutor chain...

Invoking: `Web_Search` with `LangGraph documentation`


[{'title': 'Reference - GitHub Pages', 'url': 'https://langchain-ai.github.io/langgraph/reference/', 'content': '## LangGraph Platform¶\n\nTools for deploying and connecting to the LangGraph Platform.\n\n SDK (Python): Python SDK for interacting with instances of the LangGraph Server.\n SDK (JS/TS): JavaScript/TypeScript SDK for interacting with instances of the LangGraph Server.\n RemoteGraph: `Pregel` abstraction for connecting to LangGraph Server instances.\n\nSee the LangGraph Platform reference for more reference documentation.', 'score': 0.99999547}, {'title': 'What is LangGraph? - IBM', 'url': 'https://www.ibm.com/think/topics/langgraph', 'content': '# What is LangGraph?\n\n## Authors\n\nBryan Clark \n\nSenior Technology Advocate\n\n## LangGraph overview\n\nLangGraph, created by LangChain, is an open source AI agent framework designed to build, deploy and manage complex generative AI

In [8]:
#6) OpenAI Multi-Functions 
# (can call multiple tools in one step or for one single turn)
agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent=AgentType.OPENAI_MULTI_FUNCTIONS,
    verbose=True
)
result = agent.run("Answer briefly from QA; also check the web for any updates.")



> Entering new AgentExecutor chain...

Invoking: `Simple_QA` with `{'tool_input': 'What is the capital of France?'}`


The capital of France is Paris.
Invoking: `Web_Search` with `{'tool_input': 'Capital of France'}`


[{'title': 'Capital of France | Paris Travel Guide, Facts & Highlights - SimCorner', 'url': 'https://simcorner.com/blogs/travel-guides/capital-of-france', 'content': 'Paris is the capital of France, a major European city where history, culture, and modern life meet along the Seine River. The city combines iconic landmarks like the Eiffel Tower and the Louvre Museum with bustling boulevards, gourmet bistros, and lively markets. As the political hub, housing the French President at the Élysée Palace and the National Assembly, Paris also offers a rich cultural experience through museums, theaters, and fashion districts. It is a top destination for travelers exploring tourist attractions in France or visiting Disneyland Paris. [...] Paris serves as the capital of France an

In [9]:
#7) Self-Ask with Search (clarifies, then searches)

search_tool = Tool(
    name="Intermediate Answer",  # This name is REQUIRED
    func=TavilySearchResults(max_results=1).run,
    description="Use this tool to search for missing facts"
)

agent = initialize_agent(
    tools=[search_tool],
    llm=llm,
    agent=AgentType.SELF_ASK_WITH_SEARCH,
    verbose=True,
    handle_parsing_errors=True
)
result = agent.run("When was the first LangChain release and who founded it?")




> Entering new AgentExecutor chain...
Yes. 
Follow up: Who founded LangChain?
Intermediate answer: [{'title': "LangChain's Harrison Chase on Building the Orchestration Layer for ...", 'url': 'https://sequoiacap.com/podcast/training-data-harrison-chase/', 'content': 'Sonya Huang: Hi, and welcome to training data. We have with us today Harrison Chase, founder and CEO of LangChain. Harrison is a legend in the agent ecosystem, as the product visionary who first connected LLMs with tools and actions. And LangChain is the most popular agent building framework in the AI space. Today, we’re excited to ask Harrison about the current state of agents, the future potential and the path ahead. Harrison, thank you so much for joining us. And welcome to the show.\n\nHarrison Chase: Of course, thank you for having me.\n\n#### What are agents?', 'score': 0.8377398}]
Follow up: When was the first LangChain release?
Intermediate answer: [{'title': 'LangChain - Wikipedia', 'url': 'https://en.wikipedia.o